# featurewiz, explained 4/4: Full pipeline
SULOV, then Recursive XGBoost, as `featurewiz()` runs them. We'll check our version against the
real featurewiz, score it against the planted truth, and finish on a much harder dataset.

In [ ]:
import pathlib, sys

for module_file in pathlib.Path("/kaggle/input").glob("**/featurewiz_explained.py"):
    sys.path.insert(0, str(module_file.parent))
    break

import featurewiz_explained as fwx
import featurewiz_explained_plots as plots

> **featurewiz, explained**: a four-part series that rebuilds featurewiz 0.6.1's feature selection step by step.
> 1 Intuition · 2 SULOV · 3 Recursive XGBoost · **4 Full pipeline**.
> Everything here runs with Internet **off**. Results are checked against Reference runs of the real featurewiz 0.6.1.

## Our pipeline vs the real featurewiz 0.6.1

In [ ]:
data = fwx.make_canonical_dataset()
selection = fwx.select_features(data.frame, data.target)
reference = fwx.load_reference("canonical")
comparison = fwx.compare_to_reference(selection, reference)
print("ours:      ", selection.selected)
print("featurewiz:", reference.selected)
print("pinned environment:", fwx.matches_reference_environment(reference), "| exact match:", comparison.exact)

The Reference runs were computed once with the real featurewiz 0.6.1 (and xgboost 1.7.6), because
featurewiz's dependency pins clash with Kaggle's current image. In that pinned environment our version
matches it **exactly**: every Round's gains to the last bit. Here on Kaggle, xgboost is newer, so gains
can drift; the selection above shows whether it drifted for this dataset.

## Scoring against the planted truth, by Signal group
Keeping `x3_copy` instead of `x3` isn't a mistake: both carry the same signal. So we score by
**Signal group** (an informative feature plus its copies): a perfect selection keeps exactly one
feature from each group and no noise.

In [ ]:
score = fwx.score_by_signal_group(selection.selected, data)
score

In [ ]:
plots.selection_table(selection.selected, data).show()

3 of 4 groups covered, no extras, no noise, and `x4` missed (Lesson 3 explains why).

## Breast Cancer Wisconsin

In [ ]:
bc = fwx.load_breast_cancer()
bc_selection = fwx.select_features(bc, "diagnosis")
bc_reference = fwx.load_reference("breast_cancer")
print(f"30 -> SULOV {len(bc_selection.sulov.kept)} -> {len(bc_selection.selected)}:", bc_selection.selected)
print("featurewiz 0.6.1:", bc_reference.selected)
fwx.compare_to_reference(bc_selection, bc_reference)

30 measurements become 15 after SULOV and 6 after Recursive XGBoost. No ground truth here: this is real
data, so we can only compare with the Reference run.

## Capstone: a Madelon-style dataset
The NIPS 2003 feature-selection challenge used **Madelon**: 5 informative features, 15 redundant
mixtures of them, and hundreds of noise features. Real Madelon never published which columns are which,
so we build one with the same recipe (scikit-learn's `make_classification` *is* that generator):
5 informative, 15 redundant, 280 noise. This cell takes a few minutes on Kaggle's CPUs.

In [ ]:
madelon = fwx.make_madelon_style_dataset()
m_selection = fwx.select_features(madelon.frame, madelon.target)
m_score = fwx.score_relevant_vs_noise(m_selection.selected, madelon)
print(f"SULOV dropped: {m_selection.sulov.dropped}")
print(f"selected {len(m_selection.selected)}: relevant {len(m_score.relevant_kept)}/{m_score.relevant_total}, "
      f"noise {len(m_score.noise_kept)}/{m_score.noise_total}")

In [ ]:
plots.rounds_heatmap(m_selection.xgb).show()

In [ ]:
m_reference = fwx.load_reference("madelon_style")
m_reference_score = fwx.score_relevant_vs_noise(m_reference.selected, madelon)
print(f"featurewiz 0.6.1 selected {len(m_reference.selected)}: relevant {len(m_reference_score.relevant_kept)}/20, "
      f"noise {len(m_reference_score.noise_kept)}/280")
fwx.compare_to_reference(m_selection, m_reference)

The real featurewiz 0.6.1 keeps **161 features, of which only 14 are relevant and 147 are noise**. (On a
newer xgboost our run above lands close but not identical; Lesson 3 explains why.) That follows from
Lesson 3: with 295 features each Round must nominate at least `top_num = 59`, and there are only
20 relevant features to go round, so the rest of each quota is filled with noise that got a little
gain by chance. On wide, noisy data, treat featurewiz's output as a *shortlist*, not an answer.

## Sidebar: regression
featurewiz decides a target is numeric by counting its values: more than 30 distinct integers, or
more than 15 distinct floats. For such a target it swaps in `mutual_info_regression` for SULOV and an
XGBoost regressor (`reg:squarederror`, no `num_class`) for each Round. The pairs, the knockout walk,
the windows and the nomination rule work the same way. This series sticks to classification, and
`fwx.select_features` refuses targets featurewiz would treat as regression.

## Optional: rerun the real featurewiz yourself
This needs **Internet on** (phone-verified Kaggle account) and installs featurewiz's old pins into a
separate virtual environment, so your notebook's packages stay untouched. Copy it into a cell if you want to try:

```python
!python -m venv /tmp/fw && /tmp/fw/bin/pip -q install featurewiz==0.6.1 xgboost==1.7.6 scikit-learn==1.5.2 "numpy<2" "pandas<3" "pyarrow<17"
data.frame.to_csv("/tmp/canonical.csv", index=False)
!/tmp/fw/bin/python -c "import pandas as pd, featurewiz as fw; print(fw.featurewiz(pd.read_csv('/tmp/canonical.csv'), 'target', corr_limit=0.8, verbose=0)[0])"
```

## The whole series in five lines
1. SULOV removes **linearly** redundant features, keeping the one with more mutual information.
2. Recursive XGBoost trains on **suffixes** of the column list and pools each Round's top features.
3. **Column order** (SULOV's output order) changes what Recursive XGBoost finds.
4. Each Round nominates a **fixed minimum**, so with many features, noise gets in.
5. Pin your xgboost version if you need reproducible selections.